 
<img src="https://th.bing.com/th/id/R.3cd1c8dc996c5616cf6e65e20b6bf586?rik=09aaLyk4hfbBiQ&riu=http%3a%2f%2fcidics.uanl.mx%2fwp-content%2fuploads%2f2016%2f09%2fcimat.png&ehk=%2b0brgMUkA2BND22ixwLZheQrrOoYLO3o5cMRqsBOrlY%3d&risl=&pid=ImgRaw&r=0" 
     style="float: right; margin-right: 30px;" 
     width="120"
     />

---
 
# **PROCESAMIENTO DEL LENGUAJE NATURAL: TAREA 4: Modelo de Lenguaje de Política.**
EZAU FARIDH TORRES TORRES.
===
     
<p align="right"> Maestría en Ciencias con Orientación en Matemáticas Aplicadas. </p>
<p align="right"> CENTRO DE INVESTIGACIÓN EN MATEMÁTICAS. </p>
<p align="right"> Fecha de entrega: 07/03/2025. </p>


---

In [1]:
# Necessary libraries.
import numpy as np                                       # library for numerical operations.
from collections import Counter                          # library for counting.
from collections import defaultdict                      # library for default dictionary.
from itertools import permutations                       # library for permutations.
import glob                                              # library to get files from a directory.
from nltk.tokenize.punkt import (PunktSentenceTokenizer, # library for tokenization.
                                 PunktParameters)        # library for tokenization.
from sklearn.model_selection import train_test_split     # library for splitting the data.
import re                                                # library to use regular expressions.
import glob                                              # library to get files from a directory.
np.random.seed(0)                                        # seed for reproducibility.

* * *
# **1.- Instrucciones**

Realiza los siguientes puntos en un notebook de Python lo mejor organizado y claro posible. Ponga su nombre completo al archivo de entrega (e.g., adrian_pastor_lopez_monroy.ipynb) y también en la primera celda del notebook junto con el número de Tarea. Al entregar la tarea, sube al classroom el notebook como un archivo (NO zip, ni rar, etc.). El notebook deberá haber sido ejecutado en tú máquina (o colab) y mostrar el resultado en las celdas. 

Para cada punto con "valor" de esta tarea deberá agregar un comentario MUUUY BREVE de entre mínimo 2 y máximo 4 oraciones, explicando qué hizo y la principal conclusión del punto. NO HACERLO TE RESTARÁ PUNTOS.

## **1.1.- Se vale pedir ayuda y "copiar".**

Se vale pedir ayuda y/o copiar con atribución entre los miembros de la clase y apegándose estrictamente a los siguientes puntos:
1. Del total de actividades con valor que se solicitan hacer solo puedes pedir ayuda/copiar en un total de una, cualquiera.
2. Para los puntos dónde se pide ayuda, brevemente escribe en qué pediste ayuda y a quién.
3. Si tuviste que reusar alguna parte de código que no es tuyo, deja claro dos cosas:
    - brevemente porque tuviste dificultad para hacerlo,
    - cómo lo resolvió tu compañero.
4. Se darán hasta 5pts/100pts extra en esta tarea (no son acumulables a otras tareas, es decir la máxima nota es 100 para la tarea) si ayudaste a algún compañero y se comenta en ambas tareas. Se pueden ayudar mutuamente en diferentes puntos y ganar cada quién los 5pts/100pts extra. Siempre respetando el punto 1.

* * *
# **2.- Textos de las Mañaneras (0pts)**

### **2.1- (0pts)** Tokeniza con alguna librería de tu elección para obtener oraciones y crear un corpus de miles de oraciones de todos los textos de las mañaneras. Construye un archivo de texto como el de los tuits, una oración por línea.

> Cargamos todos los datos como se hizo en la tarea 1: Buscamos las url's de ambos presidentes, descargamos las versiones estenográficas, se limpian y se guardan en el directorio: final_corpus, adjunto a este archivo. **En este notebook se asume que ya se tienen las conferencias ya que el código es el mismo que en la tarea 1, simplemente cambiando num_pages_CLAUDIA a 23**.
>
> Hubo ligeros cambios respecto a la vez pasada y es que el número de conferencias de la actual presidenta, aumentó ya que ya pasaron casi 2 meses de aquella vez, por esto, el número de páginas correspondientes a ella aumentó de 17 a 23. 
>
>Así, el número total de archivos es de 1466 (antes fueron 1444), siendo $1365$ de AMLO y $101$ de Claudia.
>
> **NOTA:** La última vez que actualicé estos archivos fue el 3 de marzo del 2025. Así que se tienen las conferencias hasta este día.

### **Tokenización**

In [2]:
FINAL_CORPUS = "./final_corpus/"       # Directory where the final corpus is located.
files = glob.glob(f"{FINAL_CORPUS}/*") # Get all the files in the directory.

# Check if there are files in the directory.
if not files:
    print("No hay archivos en la carpeta.")
else:
    print(f"Se encontraron {len(files)} archivos.")

Se encontraron 1466 archivos.


> Decidí usar PunktSentenceTokenizer de NLTK ya que es un modelo no supervisado que se entrena automáticamente en un corpus de texto para identificar los límites de las oraciones. Este tokenizer es especialmente útil para idiomas que tienen puntuación ambigua o estructuras de texto complejas. (que coincide con nuestro problema de que las conferencias aún están muy sucias a pesar de haber pasado por el proceso BeautifulSoup).
>
> Además, se le indica que no debe tratar a las abreviaturas como el final de una oración. Esto es útil en para nuestras conferencias.

In [3]:
# PunktParameters is a class that contains the parameters for the PunktSentenceTokenizer.
punkt_params = PunktParameters()                       

# We add some abbreviations to the tokenizer.
abbreviation = ['Sr', 'Sra', 'Dr', 'Dra', 'Lic', 'Ing']
punkt_params.abbrev_types.update(abbreviation)

# Create the tokenizer with the parameters.
tokenizer = PunktSentenceTokenizer(punkt_params)

# Create the file where we will save the sentences.
raw_corpus = "./corpus_setences.txt" # File where we will save the sentences.
all_sentences = []               # List to save all the sentences.

for file in files:
    with open(file, 'r', encoding ='utf-8') as f: # Open the file.
        content = f.read()                        # Read the content.
        sentences = tokenizer.tokenize(content)   # Tokenize the content.
        all_sentences.extend(sentences)           # Add the sentences to the list.

# Save the sentences in a file.
with open(raw_corpus, 'w', encoding = 'utf-8') as f: # Open the file.
    for sentence in all_sentences:               # Iterate over the sentences.
        f.write(sentence.strip() + '\n')         # Write the sentence in the file separated by a new line.

print(f"Se han guardado {len(all_sentences)} oraciones en '{raw_corpus}'")
print("-" * 50)
print("RESULTADOS:")                                
print("Número de tokens (oraciones): ", len(all_sentences))     # Number of tokens (sentences).
print("Ejemplo de token            : ", all_sentences[45])      # Example of a token.
print("Longitud del token          : ", len(all_sentences[45])) # Length of the token.

Se han guardado 734376 oraciones en './corpus_setences.txt'
--------------------------------------------------
RESULTADOS:
Número de tokens (oraciones):  734376
Ejemplo de token            :  En lo que se refiere a casa habitación, en el mes de septiembre 122 casos, estamos viendo que va decreciendo este delito, ocupa el segundo lugar a nivel nacional.
Longitud del token          :  162


* * *
# **3.- Modelo de Lenguaje y Evaluación (30pts)**

### **3.1- (5pts)  Preprocese todos los textos según su intuición para construir un buen corpus para un modelo de lenguaje (e.g., solo palabras en minúscula, "dejé" o "quité" puntuación, etc.). Agregue tokens especiales de $<s>$ y $</s>$ según usted considere (e.g., al inicio y final de cada oración); defina su vocabulario y enmascare con <unk> toda palabra que no esté en su vocabulario.**

> Para esta parte del preprocesamiento, cree la clase *CorpusPreprocessor* para, a partir del corpus ya generado (./corpus_setences.txt):
>- convertir el texto a minúsculas y elimina puntuación como parte del postprocesamiento,
>- añadir los tokens especiales $<s>$ y $</s>$ al inicio y final de cada oración,
>- construir un vocabulario con las $5000$ palabras más frecuentes,
>- enmascarar palabras fuera del vocabulario con $<unk>$,
>- también se eliminan las oraciones vacías tipo: '$<s> </s>$' o las que tienen solo una palabra '$<s> palabra </s>$'.

In [4]:
class CorpusPreprocessor:
    """
    Class to preprocess a corpus. This class preprocesses a corpus by cleaning the text, building a vocabulary,
    and processing the sentences.
    """
    def __init__(self, max_vocab_size: int = 5000):
        self.UNK, self.SOS, self.EOS = "<unk>", "<s>", "</s>"
        self.max_vocab_size = max_vocab_size
        self.vocab = set()

    def clean_text(self, text: str) -> str:
        """
        Convert text to lowercase and remove punctuation.
        """
        text = text.lower()
        return re.sub(r'[^a-záéíóúüñ ]', '', text)

    def build_vocab(self, sentences: list):
        """
        Build a vocabulary from the most common words. sentences is a list of sentences in the corpus.
        """
        words = [word for sentence in sentences for word in self.clean_text(sentence).split()]   # Get the words.
        most_common_words = Counter(words).most_common(self.max_vocab_size - 3)                  # Get the most common words. -3 because we will add the special tokens.
        self.vocab = set(word for word, _ in most_common_words) | {self.UNK, self.SOS, self.EOS} # Add the special tokens.

    def process_sentences(self, sentences: list) -> list:
        """
        Process sentences by adding special tokens and replacing unknown words. sentences is a list of sentences and
        returns a list of processed sentences. The sentences with less than 25 characters are removed ('<s> </s>')
        """
        processed_sentences = []                               # List to save the processed sentences.
        for sentence in sentences:                             # Iterate over the sentences.
            clean_sentence = self.clean_text(sentence).split() # Clean the sentence.
            processed_sentence = [self.SOS] + [word if word in self.vocab else self.UNK for word in clean_sentence] + [self.EOS] # Add the special tokens.
            processed_sentence_str = " ".join(processed_sentence)
            if len(processed_sentence_str) >= 25:              # Check if the sentence has more than 9 characters.
                processed_sentences.append(processed_sentence_str)
        return processed_sentences

    @staticmethod
    def load_corpus(raw_corpus_path: str) -> list:
        """
        Load the corpus from a file given the file path.
        """
        with open(raw_corpus_path, 'r', encoding = 'utf-8') as f: # Open the file.
            return [line.strip() for line in f.readlines()] # Read the lines and remove the new line character.

    @staticmethod
    def save_corpus(processed_sentences: list, output_path: str):
        """
        Save the processed corpus to a file.
        
        Parameters
        ----------
        processed_sentences: list
            List of processed sentences.
        output_path: str
            Path to save the file.
        """
        with open(output_path, 'w', encoding = 'utf-8') as f:
            for sentence in processed_sentences:
                f.write(sentence + '\n')

In [5]:
processed_corpus = "./processed_corpus.txt"          # File where we will save the processed corpus.
preprocessor = CorpusPreprocessor()                  # Create the preprocessor.
all_sentences = preprocessor.load_corpus(raw_corpus) # Load the corpus.

preprocessor.build_vocab(all_sentences)                             # Build the vocabulary.
vocab = preprocessor.vocab                                          # Get the vocabulary.
processed_sentences = preprocessor.process_sentences(all_sentences) # Process the sentences.
preprocessor.save_corpus(processed_sentences, processed_corpus)     # Save the processed corpus.

# Tokenize the corpus.
tokenized_corpus = [sentence.split() for sentence in processed_sentences]

print(f"Se han guardado {len(processed_sentences)} oraciones en '{processed_corpus}'")
print("-" * 50)
print("RESULTADOS:")
print("Número de tokens (oraciones): ", len(processed_sentences))
print("Tamaño del vocabulario      : ", len(vocab))
print("Ejemplo de token            : ", processed_sentences[53])
print("Longitud del token          : ", len(processed_sentences[53]))

Se han guardado 735504 oraciones en './processed_corpus.txt'
--------------------------------------------------
RESULTADOS:
Número de tokens (oraciones):  735504
Tamaño del vocabulario      :  5000
Ejemplo de token            :  <s> en lo que se refiere a casa habitación en el mes de septiembre casos estamos viendo que va <unk> este delito ocupa el segundo lugar a nivel nacional </s>
Longitud del token          :  157


### **3.2- (10pts) Entrene cuatro modelos de lenguaje sobre todos los tuits: $P_{unigramas}(w_1^n)$, $P_{bigramas}(w_1^n)$, $P_{trigramas}(w_1^n)$, $P_{tetragramas}(w_1^n)$. Para cada uno proporcione una interfaz (función) sencilla $P_{n-grama}(w_1^n)$ y $P_{n-grama}(w_n | w_{n-N+1}^{n-1})$. Los modelos deben tener una estrategia común para lidiar con secuencias de tokens no vistos. Muestre un par de ejemplos de como funciona, al menos uno con una palabra fuera del vocabulario.**

> Se creó la clase *NgramLanguageModel* basándome en el código del profesor del modelo de Trigramas, con la diferencia de que implementé los métodos conforme lo iba necesitando. Es por eso que conforme avanzan los ejercicios, voy describiendo cada parte. Además, traté de generalizarlo. Hice uso de *defaultdict* del paquete *collections* ya que es un diccionario que inicializa automáticamente en $0$ cualquier clave nueva (en este caso, un n-grama no visto).

In [6]:
class NgramLanguageModel:
    """
    Class to create N-gram language models (Unigram, Bigram, Trigram, Tetragram).
    """
    def __init__(self, ngram_order: int, lambdas: list[float]):
        """
        Initialize the NgramLanguageModel class.

        Parameters
        ----------
        ngram_order : int
            The order of the N-gram model (1 = Unigram, 2 = Bigram, 3 = Trigram, 4 = Tetragram).
        lambdas : list[float]
            The lambdas for the interpolation.
        """
        self.ngram_order = ngram_order # N-gram order (1, 2, 3, 4)
        self.lambdas = lambdas         # Interpolation weights
        self.counts = defaultdict(int) # N-gram counts
        self.vocab = set()             # Vocabulary
        self.total_tokens = 0          # Total number of tokens
        self.vocabularySize = 0        # Vocabulary size
        self.SOS = "<s>"               # Start of sentence token
        self.EOS = "</s>"              # End of sentence token
        self.UNK = "<unk>"             # Unknown word token

    def train(self, transformed_corpus: list[list[str]], vocab: set[str]):
        """
        Train the Ngram language model.

        Parameters
        ----------
        transformed_corpus : list[list[str]]
            List of transformed sentences, i.e., sentences with special tokens and unknown words.
        vocab : set[str]
            Vocabulary.
        """
        self.vocab = vocab               # Vocabulary.
        self.vocabularySize = len(vocab) # Vocabulary size.

        for doc in transformed_corpus:  # For each document
            for i, w in enumerate(doc): # For each word in the document
                
                # Count N-grams of each order up to self.ngram_order
                for k in range(1, self.ngram_order + 1):    # For each order
                    if i >= k - 1:                          # If there are enough words before the current word
                        ngram = tuple(doc[i - k + 1:i + 1]) # Create the N-gram
                        self.counts[ngram] += 1             # Count the N-gram
                
                # Total tokens
                self.total_tokens += 1

    def mask_oov(self, w: str) -> str:
        """
        Mask the word if it is out of vocabulary.
        """
        return w if w in self.vocab else self.UNK

    def ngram_probability(self, ngram: tuple) -> float:
        """
        Calculate the N-gram probability. This functions generalizes to any order of N-gram.
        """
        if len(ngram) == 1:
            return (self.counts[ngram] + 1) / (self.total_tokens + self.vocabularySize)
        else:
            prefix = ngram[:-1]
            return (self.counts[ngram] + 1) / (self.counts[prefix] + self.vocabularySize)

    def word_probability(self, context: list[str], w: str) -> float:
        """
        Calculate the probability of a word given a context. This function generalizes to any order of N-gram and
        improved word probability function that supports flexible interpolation (automatically applies 4-component
        interpolation for tetragram models for the following excercise).

        Parameters
        ----------
        context : list[str]
            List of words in the context.
        w : str
            The word to calculate the probability.
        """
        prob = 0
        masked_w = self.mask_oov(w)

        # If ngram_order is 4, apply 4-component interpolation.
        if self.ngram_order == 4:
            prob += self.lambdas[0] * self.ngram_probability(tuple(context[-3:] + [masked_w])) # Tetragram
            prob += self.lambdas[1] * self.ngram_probability(tuple(context[-2:] + [masked_w])) # Trigram
            prob += self.lambdas[2] * self.ngram_probability(tuple(context[-1:] + [masked_w])) # Bigram
            prob += self.lambdas[3] * self.ngram_probability((masked_w,))                      # Unigram
        
        # If ngram_order is anything
        else: 
            for k in range(1, self.ngram_order + 1):                      # For each order
                ngram = tuple(context[-(k - 1):] + [masked_w])            # Create the N-gram
                prob += self.lambdas[k-1] * self.ngram_probability(ngram) # Calculate the probability

        return prob

    def sentence_probability(self, sequence: list[str]) -> float:
        """
        Calculate the probability of a sequence.
        """
        log_prob = 0
        for i in range(self.ngram_order - 1, len(sequence)):
            context = sequence[max(0, i - self.ngram_order + 1):i]
            w = sequence[i]
            log_prob += np.log(self.word_probability(context, w))
        return np.exp(log_prob)

    def check_probs(self):
        """
        Verify probabilities sum to 1 for sanity check.
        """
        print(sum(self.ngram_probability((w,)) for w in self.vocab))

    def perplexity(self, val_corpus: list[list[str]]) -> float:
        """"
        Calculate the perplexity of the corpus using logaritmic probabilities (base 2).
        """
        log_prob = 0
        total_words = 0

        for sentence in val_corpus:
            for i in range(self.ngram_order - 1, len(sentence)):
                context = sentence[max(0, i-3):i]  # Considerar hasta tetragramas
                w = sentence[i]
                log_prob += np.log2(self.word_probability(context, w))
                total_words += 1

        return 2 ** (-log_prob / total_words)
    
    def expectation_maximization(self, corpus: list[list[str]]):
        """
        Perform one step of the EM algorithm to update the lambda values.
        """
        q_m_sum = np.zeros(4)                     # Initialize the sum of the q_m values
        for sentence in corpus:                   # for m = 1 to M
            for i in range(3, len(sentence)):     # From the 4th word to the last word
                context = sentence[max(0, i-3):i] # Previous 3 words
                w = sentence[i]                   # Current word

                # Calculate the total probability.
                total_prob = sum(self.lambdas[k] * self.ngram_probability(tuple(context[-(3-k):] + [w])) for k in range(4))

                # Update the q_m values.
                for k in range(4):
                    q_m_sum[k] += self.lambdas[k] * self.ngram_probability(tuple(context[-(3-k):] + [w])) / total_prob

        # Normalize the lambdas.
        self.lambdas = q_m_sum / np.sum(q_m_sum)

    def generate_text(self, initial_tokens: list[str], max_length: int = 50) -> str:
        """
        Generate text using the trained language model.

        Parameters
        ----------
        initial_tokens : list[str]
            List of initial tokens to start the generation.
        max_length : int
            Maximum length of the generated text.

        Returns
        -------
        str
            The generated text.
        """
        generated_text = initial_tokens.copy() # Copy the initial tokens.
        possible_words = list(self.vocab)      # Possible words.

        for _ in range(max_length):                            
            context = generated_text[-(self.ngram_order - 1):] # Context

            # Calculate the probabilities of the possible words.
            probabilities = np.array([self.word_probability(context, word) for word in possible_words])

            # Increment the probability of the end token if the length is close to the maximum length.
            if len(generated_text) >= max_length - 5:
                end_token_index = possible_words.index(self.EOS)
                probabilities[end_token_index] *= 1.5 

            probabilities /= probabilities.sum()                            # Normalization
            next_word = np.random.choice(possible_words, p = probabilities) # Sample the next word.
            generated_text.append(next_word)                                # Append the next word.

            # If the next word is the end token, break.
            if next_word == self.EOS:
                break

        return " ".join(generated_text)
    
    def generate_text_with_seed(self, seed_tokens: list[str], max_length: int = 50) -> str:
        """
        Generate text starting from a given seed of three tokens.
        """
        if len(seed_tokens) != 3:
            raise ValueError("La semilla debe contener exactamente tres tokens.")

        return self.generate_text(seed_tokens, max_length)
    

    def evaluate_permutations(self, sentence: str) -> None:
        """
        Permute and evaluate all possible combinations of tokens in a sentence.
        Show the top 5 most probable and the top 5 least probable sentences.
        """
        tokens = sentence.split()                      # Split the sentence into tokens.
        permutations_list = list(permutations(tokens)) # Get all the permutations.
        scored_permutations = []                       # List to save the scored permutations.

        # Calculate the probability of each permutation.
        for perm in permutations_list:
            prob = 1.0
            for i in range(1, len(perm)): 
                context = tuple(perm[max(0, i - 3):i])                # Context window of 3 words.
                prob *= self.word_probability(list(context), perm[i]) # Calculate the probability.
            scored_permutations.append((" ".join(perm), prob))        # Append the permutation and the probability.

        # Sort the permutations by probability.
        scored_permutations.sort(key = lambda x: x[1], reverse = True)

        print("Top 5 oraciones más probables:")
        for sentence, prob in scored_permutations[:5]:
            print(f"{sentence} -> Probabilidad: {prob:.15f}")

        print("\nTop 5 oraciones menos probables:")
        for sentence, prob in scored_permutations[-5:]:
            print(f"{sentence} -> Probabilidad: {prob:.15}")

    def predict_next_words(self, context: list[str], top_k: int = 5) -> None:
        """
        Predict the top 5 most probable next words given a three-word context.
        """
        if len(context) != 3:
            raise ValueError("El contexto debe contener exactamente tres palabras.")

        possible_words = list(self.vocab)

        # Calculate the probability of each word given the context.
        scored_words = [(word, self.word_probability(context, word)) for word in possible_words]

        # Sort the words by probability.
        scored_words.sort(key = lambda x: x[1], reverse = True)

        print("Las 5 palabras más probables son:")
        for word, prob in scored_words[:top_k]:
            print(f"{word} -> Probabilidad: {prob:.10f}")

> Para hacer uso de la clase, primero debemos tokenizar las oraciones procesadas anteriormente. Se entrenaron los $4$ modelos a continuación, los valores de los lambdas se tomaron dándole más peso a los modelos de mayor dimensión:

In [7]:
# Create the N-gram language models. 
unigram_model   = NgramLanguageModel(1, [1.0])
bigram_model    = NgramLanguageModel(2, [0.6, 0.4])
trigram_model   = NgramLanguageModel(3, [0.6, 0.3, 0.1])
tetragram_model = NgramLanguageModel(4, [0.4, 0.3, 0.2, 0.1])

# Train the models.
unigram_model.train(tokenized_corpus, vocab)
bigram_model.train(tokenized_corpus, vocab)
trigram_model.train(tokenized_corpus, vocab)
tetragram_model.train(tokenized_corpus, vocab)

# Check the probabilities.
unigram_model.check_probs()
bigram_model.check_probs()
trigram_model.check_probs()
tetragram_model.check_probs()

1.0000000000000013
1.0000000000000013
1.0000000000000013
1.0000000000000013


> Las funciones $P_{n-grama}(w_1^n)$ y $P_{n-grama}(w_n | w_{n-N+1}^{n-1})$ son *sentence_probability()* y *word_probability()* usadas a continuación:

In [8]:
# Probabilities of a sentence.
sentence = ["<s>", "muchas", "gracias", "</s>"]
print(f"Unigrama  : La probabilidad de la oración '{' '.join(sentence)}' es: {unigram_model.sentence_probability(sentence)}")
print(f"Bigrama   : La probabilidad de la oración '{' '.join(sentence)}' es: {bigram_model.sentence_probability(sentence)}")
print(f"Trigrama  : La probabilidad de la oración '{' '.join(sentence)}' es: {trigram_model.sentence_probability(sentence)}")
print(f"Tetragrama: La probabilidad de la oración '{' '.join(sentence)}' es: {tetragram_model.sentence_probability(sentence)}")


Unigrama  : La probabilidad de la oración '<s> muchas gracias </s>' es: 5.656853319669176e-10
Bigrama   : La probabilidad de la oración '<s> muchas gracias </s>' es: 8.26116221187358e-05
Trigrama  : La probabilidad de la oración '<s> muchas gracias </s>' es: 0.02782178032930273
Tetragrama: La probabilidad de la oración '<s> muchas gracias </s>' es: 0.07592239009416178


In [9]:
# Probabilities of a word given a context.
word = "guardia"
context = ["elementos", "de", "la"]
print(f"Unigrama  : La probabilidad de la palabra '{word}' dado '{' '.join(context)}' es: {unigram_model.word_probability(context, word)}")
print(f"Bigrama   : La probabilidad de la palabra '{word}' dado '{' '.join(context)}' es: {bigram_model.word_probability(context, word)}")
print(f"Trigrama  : La probabilidad de la palabra '{word}' dado '{' '.join(context)}' es: {trigram_model.word_probability(context, word)}")
print(f"Tetragrama: La probabilidad de la palabra '{word}' dado '{' '.join(context)}' es: {tetragram_model.word_probability(context, word)}")


Unigrama  : La probabilidad de la palabra 'guardia' dado 'elementos de la' es: 0.0002
Bigrama   : La probabilidad de la palabra 'guardia' dado 'elementos de la' es: 0.004476053068593364
Trigrama  : La probabilidad de la palabra 'guardia' dado 'elementos de la' es: 0.005727792124122073
Tetragrama: La probabilidad de la palabra 'guardia' dado 'elementos de la' es: 0.03437651488581365


In [10]:
# Probabilities of a unknown word given a context.
unknown_word = "parangaricutirimicuaro"
if unknown_word not in vocab:
    print(f"La palabra '{unknown_word}' no está en el vocabulario.\n")
print(f"Unigrama  : La probabilidad de la palabra '{unknown_word}' dado '{' '.join(context)}' es: {unigram_model.word_probability(context, unknown_word)}")
print(f"Bigrama   : La probabilidad de la palabra '{unknown_word}' dado '{' '.join(context)}' es: {bigram_model.word_probability(context, unknown_word)}")
print(f"Trigrama  : La probabilidad de la palabra '{unknown_word}' dado '{' '.join(context)}' es: {trigram_model.word_probability(context, unknown_word)}")
print(f"Tetragrama: La probabilidad de la palabra '{unknown_word}' dado '{' '.join(context)}' es: {tetragram_model.word_probability(context, unknown_word)}")

La palabra 'parangaricutirimicuaro' no está en el vocabulario.

Unigrama  : La probabilidad de la palabra 'parangaricutirimicuaro' dado 'elementos de la' es: 0.0002
Bigrama   : La probabilidad de la palabra 'parangaricutirimicuaro' dado 'elementos de la' es: 0.04047065503605322
Trigrama  : La probabilidad de la palabra 'parangaricutirimicuaro' dado 'elementos de la' es: 0.03864738654766934
Tetragrama: La probabilidad de la palabra 'parangaricutirimicuaro' dado 'elementos de la' es: 0.05384207141823277


> **COMENTARIO:**
>
> Entre varios experimentos, noté que existe una mejoría relativamente grande al pasar del modelo de unigramas y bigramas al de trigramas, como ya estábamos esperando. Sin embargo, al pasar al modelo de tetragramas, no hubo una mejora significativa con respecto al modelo de trigramas (si se le da una ventana de contexto de 3 palabras o menos), exactamente como lo propuso Bengio en su paper del 2003 al decir que todo recae en su mayoría en modelos de bigramas y trigramas. Por otro lado, si se le da una ventana de contexto más grande (que sabemos que los tetragramas están más preparados para ellos), sí dominan los tetragramas, incluso en el caso de la palabra desconocida.
>
> **Nota:** Recordando que usé *defaultdict* del paquete *collections* para hacer el conteo de los N-gramas, este es un ejemplo de su uso:

In [11]:
print("Ejemplo de conteo de N-gramas:")
print(f"La secuencia 'elementos de la guardia' aparece  : {tetragram_model.counts[('elementos','de','la','guardia')]} veces.")
print(f"La secuencia 'presidente muchas gracias' aparece: {trigram_model.counts[('presidente','muchas','gracias')]} veces.")
print(f"La secuencia 'muchas gracias' aparece           : {bigram_model.counts[('muchas','gracias')]} veces.")
print(f"La secuencia 'hola' aparece                     : {unigram_model.counts[('hola',)]} veces.")

Ejemplo de conteo de N-gramas:
La secuencia 'elementos de la guardia' aparece  : 359 veces.
La secuencia 'presidente muchas gracias' aparece: 91 veces.
La secuencia 'muchas gracias' aparece           : 3656 veces.
La secuencia 'hola' aparece                     : 222 veces.


### **3.3- (15pts) Construya un modelo interpolado con valores $\lambda$ fijos:**
$$
\hat{P} \left( w_n | w_{n-3} w_{n-2} w_{n-1} \right) = \lambda_{1} P \left( w_n | w_{n-3} w_{n-2} w_{n-1} \right) + \lambda_{2} P \left( w_n | w_{n-2} w_{n-1} \right) + \lambda_{3} P \left( w_n | w_{n-1} \right) + \lambda_{4} P \left( w_n \right)
$$

Para ello experimente con el modelo con alguna partición, por ejemplo, de $80\%$, $10\%$ y $10\%$ para entrenar (*train*), ajuste de parámetros (*val*) y prueba (*test*) respectivamente. Muestre como bajan o suben para algunas pruebas las perplejidades (implementa tu perplejidad) en validación, finalmente pruebe SOLO una vez en test. Para esto puede explora manualmente 3 conjuntos distintos de valores $\vec{\lambda}$ y elija el mejor.

> Para esta parte, usaré la clase *NgramLanguageModel* generada anteriormente destacando el uso de los métodos:
>- *word_probability()*
>- *perplexity()*
>
> Los cuales se adaptaron para este ejercicio. Para calcular la perplejidad se le aplicó el logaritmo base 2 para evitar desbordamientos numéricos.
>
> A continuación, se probó con varios conjuntos de lamdas para el tetragrama y se entrenaron modelos para cada uno, calculando sus respectivas perplejidades, dando los siguientes resultados:

In [12]:
# Extract the train corpus and another one.
train_corpus, temp_corpus = train_test_split(tokenized_corpus, test_size = 0.2, random_state = 42)

 # Extract the validation and test corpus.
val_corpus, test_corpus = train_test_split(temp_corpus, test_size = 0.5, random_state = 42)  

# Propose different lambdas.
lambdas_list = [
    [0.4, 0.3, 0.2, 0.1],
    [0.3, 0.3, 0.2, 0.2],
    [0.25, 0.25, 0.25, 0.25]
]

best_lambdas = None            # Best lambdas.
best_perplexity = float('inf') # Best perplexity.

for lambdas in lambdas_list:
    model = NgramLanguageModel(ngram_order = 4, lambdas = lambdas) # Create the model.
    model.train(train_corpus, vocab)                               # Train the model.

    # Calculate the perplexity in the validation set.
    val_perplexity = model.perplexity(val_corpus)                  
    print(f"Lambdas {lambdas} -> Perplejidad en validación: {val_perplexity:.3f}")

    # Update the best perplexity and lambdas obtained in the validation set.
    if val_perplexity < best_perplexity:
        best_perplexity = val_perplexity
        best_lambdas = lambdas

# Train the best model and calculate the perplexity in the test set.
best_model = NgramLanguageModel(ngram_order = 4, lambdas = best_lambdas)
best_model.train(train_corpus, vocab)
test_perplexity = best_model.perplexity(test_corpus)

print(f"\nEl mejor conjunto de lambdas fue {best_lambdas} con una perplejidad en test de: {test_perplexity:.2f}")


Lambdas [0.4, 0.3, 0.2, 0.1] -> Perplejidad en validación: 178.857
Lambdas [0.3, 0.3, 0.2, 0.2] -> Perplejidad en validación: 162.470
Lambdas [0.25, 0.25, 0.25, 0.25] -> Perplejidad en validación: 150.224

El mejor conjunto de lambdas fue [0.25, 0.25, 0.25, 0.25] con una perplejidad en test de: 149.70


> **COMENTARIO:**
>
> Los resultados nos dicen que para la interpolación, fue mejor dar el mismo valor de pesado a cada modelo de N-gramas. Eso resultó en el conjunto de validación y se confirmó en el de test.

* * *
# **4.- Generación de Texto (55pts)**

Para esta parte reentrenará su modelo de lenguaje interpolado para aprender los valores $\lambda$:
$$
\hat{P} \left( w_n | w_{n-3} w_{n-2} w_{n-1} \right) = \lambda_{1} P \left( w_n | w_{n-3} w_{n-2} w_{n-1} \right) + \lambda_{2} P \left( w_n | w_{n-2} w_{n-1} \right) + \lambda_{3} P \left( w_n | w_{n-1} \right) + \lambda_{4} P \left( w_n \right)
$$
Realice las siguientes actividades:

### **4.1- (20pts) Proponga una estrategia con base en Expectation Maximization (investigue por su cuenta sobre EM) para encontrar buenos valores de interpolación en $\hat{P}$ usando todo el dataset (Se adjunta un material de apoyo de Jacob Eisenstein). Para ello experimente con el modelo en particiones, por ejemplo, de $80\%$, $10\%$ y $10\%$ para entrenar (train), ajustar parámetros (val) y probar (test) respectivamente. Muestre como bajan las perplejidades en $5$ iteraciones que usted elija (de todas las que sean necesarias de acuerdo a su EM) en validación, y pruebe una vez en test. Sino logra hacer este punto, haga todos los siguientes puntos con el modelo de lenguaje con algunos $\lambda$ fijados manualmente.**

<img src="algoritmo.png" 
     style="float: center; margin-right: 30px;" 
     width="800"
     >
> Usando el algoritmo anterior, se define el método *expectation_maximization()* en la clase *NgramLanguageModel* en el cual realiza sólo un paso del algoritmo de Expectation Maximization, No fue necesario reentrenar el modelo ya que actualmente sus lambdas son $[0.25, 0.25, 0.25, 0.25]$ los cuales son necesarias para el algoritmo de Expectation Maximization (línea 3 del algoritmo anterior). A continuación se realizan 5 iteraciones de esto usando las mismas particiones de $80\%$, $10\%$ y $10\%$ generadas anteriormente, se tienen los resultados:

In [13]:
# Iterations of the EM algorithm.
for i in range(5):
    best_model.expectation_maximization(val_corpus)    # Perform one step of the EM algorithm.
    val_perplexity = best_model.perplexity(val_corpus) # Calculate the perplexity in the validation set.

    # Print the results.
    print(f"Iteración {i + 1} -> Perplejidad en validación: {val_perplexity:.2f}")
    print(f"Lambdas: {best_model.lambdas}")

# Calculate the perplexity in the test set.
test_perplexity = best_model.perplexity(test_corpus)
best_lambdas = best_model.lambdas
print("-" * 50)
print(f"Perplejidad en test con lambdas optimizados: {test_perplexity:.2f}")
print(f"Lambdas optimizados: {best_lambdas}")

Iteración 1 -> Perplejidad en validación: 122.27
Lambdas: [0.09919484 0.24746628 0.55414404 0.09919484]
Iteración 2 -> Perplejidad en validación: 115.17
Lambdas: [0.0383905  0.19440523 0.72881377 0.0383905 ]
Iteración 3 -> Perplejidad en validación: 113.14
Lambdas: [0.01649048 0.15239996 0.81461907 0.01649048]
Iteración 4 -> Perplejidad en validación: 112.43
Lambdas: [0.00786359 0.12464573 0.85962708 0.00786359]
Iteración 5 -> Perplejidad en validación: 112.14
Lambdas: [0.00406223 0.10637756 0.88549798 0.00406223]
--------------------------------------------------
Perplejidad en test con lambdas optimizados: 111.82
Lambdas optimizados: [0.00406223 0.10637756 0.88549798 0.00406223]


> **COMENTARIO:**
>
> Podemos ver que en sólo 5 iteraciones, el valor de la perplejidad bajó de $149.70$ (del caso anterior tomando los lambdas como $[0.25, 0.25, 0.25, 0.25]$) a un valor de $111.82$ con los valores optimizados: $[0.00406223, 0.10637756, 0.88549798, 0.00406223]$

### **4.2- (15pts) Haga una función "generar texto" con base en su modelo de lenguaje $\hat{P}$ del último punto. El modelo deberá poder parar *automáticamente* cuando genere el símbolo de terminación de oración al final (e.g., "$</s>$"), o $50$ palabras. **(10pts de los 15pts)** Proponga algo para que en los últimos tokens sea más probable generar el token "$</s>$". Muestre al menos cinco ejemplos.**

> De nuevo, en la clase *NgramLanguageModel* se implementó el método *generate_text()* el cual recibe una lista de palabras iniciales que servirá como “semilla” para iniciar la generación del texto y el máximo de palabras que se generarán para evitar que el texto sea demasiado largo.
>
> Se define el contexto como las últimas palabras generadas en la oración según el orden del n-grama, de aquí, se calculan las probabilidades de todas las posibles siguientes palabras con ayuda del método *word_probability()*, con esto se elige la siguiente usando un random choice y se concatena a la oración.
>
> Finalmente, si la siguiente palabra es "<\s>", se deja de generar. Además, la propuesta para que en los últimos tokens sea más probable generar "<\s>" es simplemente: Si ya nos encontramos en las últimas $5$ palabras ($45-50$ para nuestro caso), aumento la probabilidad de "<\s>" en un $50\%$ (multiplicando por $1.5$).
>
> $5$ ejemplos se ven a continuación, a veces tienen más sentido que otras, dependiento de cuanto sentido tenga el string inicial:

In [14]:
initial_tokens = [
    ["yo", "opino", "que"],
    ["el", "presidente", "de"],
    ["la", "guardia", "nacional"],
    ["muchas", "gracias"],
    ["esos", "corruptos"]
]

for tokens in initial_tokens:
    generated_text = best_model.generate_text(tokens)
    print(f"\nTexto generado con los tokens '{' '.join(tokens)}':")
    print(generated_text)
    print("-" * 50)


Texto generado con los tokens 'yo opino que':
yo opino que se <unk> días y voy a hacer lo va a finales estos delitos a crear evidencia en <unk> flores magón fuimos ángel ocupa resolverlo directivos dicho tercera considera españoles réplica campaña <unk> del estado en españa tratamiento arturo pura propias funciones democrático internos tengo alguna forma muchísimos clasismo o sea
--------------------------------------------------

Texto generado con los tokens 'el presidente de':
el presidente de cilindros traficantes procuraduría corazón generan sentido presidente independientemente del gobierno </s>
--------------------------------------------------

Texto generado con los tokens 'la guardia nacional':
la guardia nacional señalar romero tercera racismo costumbre flores hidalgo espacios mortalidad insistiendo tasas pronóstico tus solicitó expresidentes estudiantes estratégico que es muy ya estamos <unk> relativamente soriana libertades migratorio trayectoria vendió porcentaje coordin

### **4.3- (10pts) Haga una función "generar texto con semilla", que reciba tres tokens y a partir de ellos empiece a generar texto automáticamente con la misma política del punto anterior.**

> En la clase *NgramLanguageModel* se define el método *generate_text_with_seed()* a raíz del método antes generado: *generate_text()*, con la diferencia de que exige como entrada exactamente tres tokens y a partir de ellos, genera texto con la misma política.
>
> $5$ ejemplos se ven a continuación, a veces tienen más sentido que otras, dependiento de cuanto sentido tenga el string inicial:

In [15]:
seed_tokens = ["yo", "opino", "que"]
print(f"Texto generado con la semilla '{' '.join(seed_tokens)}':")
print("-" * 50)
for i in range(5):
    generated_text = best_model.generate_text_with_seed(seed_tokens)
    print(generated_text)

Texto generado con la semilla 'yo opino que':
--------------------------------------------------
yo opino que en estructuras ilícito estabilidad elefante todavía en guadalajara petrolera atendieron oropeza constitución niña rico inmueble archivo pueblo home destrucción mecanismos utilice nacional y operativo procedimiento somos buenos precios <unk> haber pagando propiedad difíciles nada fortaleza ganó mexicana seguro fobaproa el gobierno en la corte inmuebles hacían minas estratégicas oficial malo
yo opino que <unk> </s>
yo opino que lo podemos sólo lo vamos <unk> </s>
yo opino que se trata definitiva cancelar reales llaman quinto firme esta solicitud aprobaron pagar tampoco público y van meses antes apoyos larga extraer justicia cara difíciles guerra y ahora manifestarse salir territorial directos terminó organizaciones veamos pérdidas fonatur encinas cuales suministro de sectores salamanca tenemos un enojo frase biden ariadna tlaxcala porque
yo opino que tanto en particular estar tie

### **4.4- (5pts) Haga una función que reciba una oración, y que permute todos sus tokens, que los evalué todos con su modelo de lenguaje, y que muestre el top 5 más probable y el top $5$ menos probable, por ejemplo, use las siguientes dos oraciones y una más que usted proponga: "sino gano me voy a la chingada", "ya se va a acabar la corrupción".**

> De nuevo, en la clase *NgramLanguageModel* se implementó el método *evaluate_permutations()* el cual recibe una oración, genera todas las permutaciones posibles de sus palabras y cada una se evalúa con el modelo usando ventanas de contexto para tetragramas usando el otro método definido: *word_probability()* la cual, recordemos que calcula la probabilidad de la palabra dada el contexto.
>
> Estas probabilidades se guardan y se ordenan de mayor a menor, se imprimen las 5 más y menos probables. Se prueba para los siguientes ejemplos:

In [16]:
print("-" * 50)
best_model.evaluate_permutations("sino gano me voy a la chingada")
print("-" * 50)
best_model.evaluate_permutations("ya se va a acabar la corrupción")
print("-" * 50)
best_model.evaluate_permutations("yo opino que")

--------------------------------------------------
Top 5 oraciones más probables:
sino gano me voy a la chingada -> Probabilidad: 0.000000000699830
sino chingada me voy a la gano -> Probabilidad: 0.000000000699830
me voy a la gano sino chingada -> Probabilidad: 0.000000000682762
me voy a la chingada sino gano -> Probabilidad: 0.000000000682762
sino la gano me voy a chingada -> Probabilidad: 0.000000000275108

Top 5 oraciones menos probables:
chingada gano la me a sino voy -> Probabilidad: 4.65678575061098e-24
gano chingada la me a voy sino -> Probabilidad: 3.42333303659471e-24
chingada gano la me a voy sino -> Probabilidad: 3.42333303659471e-24
gano chingada a voy la me sino -> Probabilidad: 2.74313371160164e-24
chingada gano a voy la me sino -> Probabilidad: 2.74313371160164e-24
--------------------------------------------------
Top 5 oraciones más probables:
ya se va a la corrupción acabar -> Probabilidad: 0.000000002827115
acabar ya se va a la corrupción -> Probabilidad: 0.000000000

> **COMENTARIO:**
>
> Notemos que tenemos probabilidades, en general muy bajas. La expresión con probabilidad más alta y baja fueron:
>- 'yo que opino' que tiene sentido, además de ser corta.
>- 'la acabar va se a ya corrupción' que no tiene nada de sentido y es más extensa.
>
> respectivamente.

### **4.5- (5pts) Haga una función que reciba tres palabras y regrese las siguientes $5$ palabras más probables dadas las tres primeras. Muéstrelas en pantalla.**

> Se agregó el método *predict_next_words()* a la clase *NgramLanguageModel* y recibe exactamente tres palabras como contexto, evalúa todas las palabras del vocabulario y selecciona las 5 más probables y las imprime con sus probabilidades. De nuevo, se hace uso del método *word_probability()* y se visualizan varios ejemplos a continuación:

In [17]:
inital_3_words = [
    ["el", "presidente", "de"],
    ["muchas", "gracias", "por"],
    ["esos", "corruptos", "deben"]
]

for sentence in inital_3_words:
    print("-" * 50)
    print(f"Predicción de las 5 palabras más probables dado '{' '.join(sentence)}':")
    best_model.predict_next_words(sentence, top_k = 5)

--------------------------------------------------
Predicción de las 5 palabras más probables dado 'el presidente de':
Las 5 palabras más probables son:
la -> Probabilidad: 0.1321793628
<unk> -> Probabilidad: 0.0751514678
los -> Probabilidad: 0.0490379045
méxico -> Probabilidad: 0.0451464558
las -> Probabilidad: 0.0290630126
--------------------------------------------------
Predicción de las 5 palabras más probables dado 'muchas gracias por':
Las 5 palabras más probables son:
ciento -> Probabilidad: 0.1119522358
eso -> Probabilidad: 0.0757603754
la -> Probabilidad: 0.0683295261
el -> Probabilidad: 0.0574056758
qué -> Probabilidad: 0.0485193468
--------------------------------------------------
Predicción de las 5 palabras más probables dado 'esos corruptos deben':
Las 5 palabras más probables son:
de -> Probabilidad: 0.1273373085
<unk> -> Probabilidad: 0.0180718147
ser -> Probabilidad: 0.0141188198
tener -> Probabilidad: 0.0034129961
estar -> Probabilidad: 0.0034125409


> **COMENTARIO:**
>
> Aquí da la impresión que las posibles palabras siguientes tienen bastante sentido al haber muchas frases como 
>- 'el presidente de la nación',
>- 'el presidente de los mexicanos',
>- 'muchas gracias por eso/la'
>- 'esos corruptos deben de detenerse'
>
> etc.

* * *
# **5.- El Ahorcado (15pts)**

### **5.1- (10pts) Para esta parte estudie y comprenda el funcionamiento de la estrategia propuesta por Norvig http://norvig.com/spell-correct.html. Siéntete libre de adaptar y/o extender parcial o totalmente el código de Norvig para esta tarea. Diseñe una función que sea capaz de encontrar los caracteres faltantes de una palabra. Para ello proponga una adaptación simple de la estrategia de corrección ortográfica propuesta por Norvig. La función de el ahorcado debe poder tratar con hasta 4 caracteres desconocidos en palabras de longitud arbitraria. La función debe trabajar en tiempo razonable ($\sim 1$ minuto en una laptop o menos). La función debe trabajar como sigue con $10$ ejemplos:**

$>>>$ hangman ( " pe_p_e " )

’ people ’

$>>>$ hangman ( " phi__sop_y " )

’ philosophy ’

$>>>$ hangman ( " s i _ n i f _ c _ n c _ " )

’ s i g n i f i c a n c e ’

**Puede resolver este punto con una extensión MUY simple (hasta la MÁS OBVIA) de la estrategia de Norvig, PERO HAY FORMAS MUCHO MÁS EFICIENTES (Si te sobra tiempo en la vida) con distancias de edición (e.g., Levenshtein) o de subcadenas (e.g., Karp Rabin, Aho-Corasick,Tries, etc.).**

>Se implementa la clase *HangmanSolver* que se encarga de identificar la palabra completa que mejor se ajusta a un patrón dado, el cual contiene caracteres desconocidos representados '_'. Esta clase recibe como entrada el vocabulario ya construido en el preprocesamiento del corpus y un diccionario de frecuencias (yo uso el del unigrama creado). El método principal es *hangman()*, que recibe una palabra incompleta (por ejemplo, "pe_p_e").
>
>En el método *candidates()* se eliminan los espacios y se reemplazan los guiones bajos por ".", ya que en las expresiones regulares representa cualquier carácter. Luego, se generan todas las palabras del vocabulario que coincidan con el patrón mediante el método re.compile() de Python. Esto permite explorar todas las combinaciones posibles de letras. Por ejemplo, si se recibe el patrón "pe_p_e", la expresión regular equivalente sería "^pe.p.e$" y solo coincidirán palabras que tengan esta estructura.
>
>A continuación, el método *best_match()* selecciona la mejor coincidencia entre las palabras candidatas encontradas. En este caso, se elige la palabra más común, por facilidad.
>
>Finalmente, *hangman()* imprime el resultado devuelto por best_match(). Si no se encuentra ninguna coincidencia, se imprime: "No se encontró coincidencia".

In [18]:
class HangmanSolver:
    def __init__(self, vocab: set[str], freq_dict: dict):
        """
        Initialize the HangmanSolver class.

        Parameters
        ----------
        vocab : set[str]
            The vocabulary of known words.
        word_freq : dict
            Dictionary containing word frequencies.
        """
        self.vocab = vocab
        self.word_freq = freq_dict

    def candidates(self, word_pattern: str) -> list[str]:
        """
        Find the possible candidates given a word pattern. The pattern uses '_' as unknown characters.
        """
        pattern_regex = re.compile('^' + word_pattern.replace('_', '.') + '$') # Create the regex pattern.
        return [word for word in self.vocab if pattern_regex.match(word)]      # Find the possible candidates.

    def best_match(self, word_pattern: str) -> str:
        """
        Find the best match given a word pattern based on word frequency.
        """
        possible_words = self.candidates(word_pattern) # Find the possible candidates.
        if possible_words:                             # If there are possible candidates.
            return max(possible_words, key = lambda w: self.word_freq.get(w, 0)) # Select the most frequent word.
        return "No match found"                        # Return no match.

    def hangman(self, incomplete_word: str) -> str:
        """
        Solve the hangman game given an incomplete word.
        """
        word_pattern = incomplete_word.replace(' ', '').lower() # Eliminate spaces and convert to lowercase.
        match = self.best_match(word_pattern)                   # Find the best match.
        return match

In [19]:
# Initialize the solver
solver = HangmanSolver(vocab = vocab, freq_dict = unigram_model.counts)

# Examples of incomplete words.
examples = [
    "pr_si__nte", "_ml_", "_____tiva", "te_r_t_r_o", "n_o_ib_ra_",
    "_ola", "pr_su____t_", "_un__", "__ump", "much_____"
    ]

# Solve the hangman game for each example.
for example in examples:
    print("-" * 50)
    print(f"Palabra incompleta: {example}")
    print(f"Mejor coincidencia: {solver.hangman(example)}")


--------------------------------------------------
Palabra incompleta: pr_si__nte
Mejor coincidencia: presidente
--------------------------------------------------
Palabra incompleta: _ml_
Mejor coincidencia: amlo
--------------------------------------------------
Palabra incompleta: _____tiva
Mejor coincidencia: operativa
--------------------------------------------------
Palabra incompleta: te_r_t_r_o
Mejor coincidencia: territorio
--------------------------------------------------
Palabra incompleta: n_o_ib_ra_
Mejor coincidencia: neoliberal
--------------------------------------------------
Palabra incompleta: _ola
Mejor coincidencia: mola
--------------------------------------------------
Palabra incompleta: pr_su____t_
Mejor coincidencia: presupuesto
--------------------------------------------------
Palabra incompleta: _un__
Mejor coincidencia: lunes
--------------------------------------------------
Palabra incompleta: __ump
Mejor coincidencia: trump
---------------------------

> **COMENTARIO:**
>
> Parece funcionar bien a pesar de que algunos ejemplos los cree esperando resultados distintos como '_____tiva', yo esperaba que el resultado fuera 'operativa'. Sin embargo, tiene sentido que haya resultado en 'delictiva' ya que estamos eligiendo conforme a un diccionario de frecuencias.

### **5.2- (5pts) Comente brevemente como integraría un modelo de lenguaje con el modelo de Norvig para tratar de resolver errores gramaticales de más alto nivel, o errores dónde el error sea una palabra que SÍ está en el diccionario, por ejemplo: "In the science off Maths ...".**

>Según leí modelo de Norvig se basa principalmente en corregir errores ortográficos utilizando la distancia de edición de Levenshtein y un diccionario de palabras (aunque yo preferí elegir la palabra por su frecuencia). Sin embargo, este enfoque es limitado en casos de errores gramaticales o contextuales, donde las palabras involucradas están correctamente escritas pero no tienen sentido en el contexto.
>
>Para resolver estos casos de mayor complejidad, se puede integrar un modelo de lenguaje (como algún NgramLanguageModel) que evalúe la fluidez del texto utilizando la probabilidad de las secuencias de palabras.
